In [1]:
from datasets import load_dataset
from datasets import Dataset, DatasetDict
from torch.utils.data import DataLoader
from transformers import DataCollatorForTokenClassification
from datasets import load_dataset
from transformers import AutoTokenizer, BertForTokenClassification,AutoModelForTokenClassification
import os
import requests
import pandas as pd
hf_token = os.getenv('HF_TOKEN')


## Per allenare il NER utilizzeremo il dataset KIND https://github.com/dhfbk/KIND/tree/main

In [2]:
def read_kind_conll(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        sentences = []
        tokens = []
        tags = []
        
        for line in f:
            line = line.strip()
            if not line: # Riga vuota = fine frase
                if tokens:
                    sentences.append({"tokens": tokens, "ner_tags": tags})
                    tokens = []
                    tags = []
            else:
                # Divide il token dalla label (gestisce spazi o tab)
                parts = line.split()
                if len(parts) >= 2:
                    tokens.append(parts[0])
                    tags.append(parts[-1])
        
        # Aggiunge l'ultima frase se il file non finisce con riga vuota
        if tokens:
            sentences.append({"tokens": tokens, "ner_tags": tags})
            
    return sentences


In [3]:
sentences = []
for file in os.listdir('data/ner/'):
    sentences.extend(read_kind_conll('data/ner/'+file))

In [4]:
print(f'Numero di frasi:{len(sentences)}')
print('-'*100)
print(f'Esempio: {sentences[0]}')

Numero di frasi:37765
----------------------------------------------------------------------------------------------------
Esempio: {'tokens': ['Il', 'ministro', 'degli', 'Esteri', 'al', 'commissario', 'capo', 'della', 'Commissione', 'alleata'], 'ner_tags': ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'ORG', 'ORG']}


### Per un addestramento NER fatto a regola d'arte, è fondamentale distinguere tra l'inizio di un'entità (Begin) e il suo seguito (Inside), specialmente per nomi composti come "Governo Militare Alleato".

In [5]:
def convert_to_iob2(tags):
    new_tags = []
    for i, tag in enumerate(tags):
        if tag == "O":
            new_tags.append("O")
        else:
            # Se è il primo token o se il tag precedente era diverso, è un Begin (B-)
            if i == 0 or tags[i-1] != tag:
                new_tags.append(f"B-{tag}")
            else:
                # Se il tag precedente è identico, è un Inside (I-)
                new_tags.append(f"I-{tag}")
    return new_tags

# Applichiamolo ai dati caricati
for s in sentences:
    s['ner_tags_splitted'] = convert_to_iob2(s['ner_tags'])

# Ora vediamo la nuova lista di tag unici
unique_tags = sorted(list(set(tag for s in sentences for tag in s['ner_tags_splitted'])))
tag2id = {tag: i for i, tag in enumerate(unique_tags)}
id2tag = {i: tag for tag, i in tag2id.items()}

print("Nuovi Tag IOB2:", unique_tags)
print(id2tag)

Nuovi Tag IOB2: ['B-LOC', 'B-ORG', 'B-PER', 'I-LOC', 'I-ORG', 'I-PER', 'O']
{0: 'B-LOC', 1: 'B-ORG', 2: 'B-PER', 3: 'I-LOC', 4: 'I-ORG', 5: 'I-PER', 6: 'O'}


In [11]:
sentences[0]

{'tokens': ['Il',
  'ministro',
  'degli',
  'Esteri',
  'al',
  'commissario',
  'capo',
  'della',
  'Commissione',
  'alleata'],
 'ner_tags': ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'ORG', 'ORG'],
 'ner_tags_splitted': ['O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'B-ORG',
  'I-ORG']}

## Carichiamo il tokenizer e il modello di base e allineiamo le etichette ai token
Dobbiamo far si che se una parola si divide in più token, il primo token avrà la corrispondente label mentre gli altri token avranno label_id = -100 per essere trascurati nel calcolo della loss

In [8]:
model_name = 'dbmdz/bert-base-italian-xxl-cased'

In [9]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [10]:
model = AutoModelForTokenClassification.from_pretrained(
    model_name, 
    num_labels=len(unique_tags),
    id2label=id2tag,
    label2id=tag2id
)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at dbmdz/bert-base-italian-xxl-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True) # con is_split_into_words stiamo dicendo al tokenizer che abbiamo già pulito i dati e splittato i testi in parole
    labels = []
    
    for i, label in enumerate(examples["ner_tags_splitted"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100) # Token speciali come [CLS]
            elif word_idx != previous_word_idx:
                # È un nuovo token, usiamo l'ID della label mappata
                label_ids.append(tag2id[label[word_idx]])
            else:
                # È un sub-token di una parola già vista, mettiamo -100
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

## Definiamo dataset e dataloader

In [14]:
train_dataset = Dataset.from_list(sentences[:30000])
test_dataset = Dataset.from_list(sentences[30000:])

# Li uniamo in un unico oggetto per comodità
raw_datasets = DatasetDict({
    'train': train_dataset,
    'test': test_dataset
})
# Applichiamo la funzione di allineamento definita nei passaggi precedenti
tokenized_datasets = raw_datasets.map(
    tokenize_and_align_labels, 
    batched=True,
    remove_columns=raw_datasets["train"].column_names
)



# 1. Settiamo il formato per PyTorch (converte le colonne in tensori)
tokenized_datasets.set_format("torch")

# 2. Inizializziamo il DataCollator (fondamentale per il padding dei batch semplicemente padda i batch e mette a -100 gli id dei token paddati
data_collator = DataCollatorForTokenClassification(tokenizer)

# 3. Definiamo i DataLoader
train_dataloader = DataLoader(
    tokenized_datasets["train"], 
    shuffle=True, 
    batch_size=16, 
    collate_fn=data_collator
)

eval_dataloader = DataLoader(
    tokenized_datasets["test"], 
    batch_size=16, 
    collate_fn=data_collator
)


Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7765 [00:00<?, ? examples/s]

## Definiamo iperparametri ottimzzatore e scheduler

In [15]:
from torch.optim import AdamW
import torch
from transformers import get_linear_schedule_with_warmup

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)

num_epochs = 3
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=50,
    num_training_steps=num_training_steps
)

In [16]:
import evaluate
metric = evaluate.load("seqeval")

In [17]:
from tqdm import tqdm
for epoch in range(num_epochs):
    # ========================================
    #            FASE DI TRAINING
    # ========================================
    model.train()
    total_train_loss = 0
    
    print(f"\n--- Epoca {epoch + 1} / {num_epochs} ---")
    train_loop = tqdm(train_dataloader, desc="Training")
    
    for batch in train_loop:
        batch = {k: v.to(device) for k, v in batch.items()}
        
        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
        
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        
        total_train_loss += loss.item()
        train_loop.set_postfix(loss=loss.item())

    avg_train_loss = total_train_loss / len(train_dataloader)
    print(f"Loss media Training: {avg_train_loss:.4f}")

    # ========================================
    #           FASE DI VALUTAZIONE
    # ========================================
    model.eval()
    all_predictions = []
    all_labels = []
    
    eval_loop = tqdm(eval_dataloader, desc="Valutazione")
    
    for batch in eval_loop:
        batch = {k: v.to(device) for k, v in batch.items()}
        
        with torch.no_grad():
            outputs = model(**batch)
        
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=-1)
        labels = batch["labels"]

        # Portiamo i dati su CPU e convertiamo in liste Python
        predictions = predictions.detach().cpu().numpy()
        labels = labels.detach().cpu().numpy()

        # Pulizia dei -100 e conversione in etichette testuali (B-PER, I-LOC, ecc.)
        for prediction, label in zip(predictions, labels):
            true_preds = [id2tag[p] for (p, l) in zip(prediction, label) if l != -100]
            true_labels = [id2tag[l] for (p, l) in zip(prediction, label) if l != -100]
            
            all_predictions.append(true_preds)
            all_labels.append(true_labels)

    # Calcolo delle metriche finali per l'epoca
    
    results = metric.compute(predictions=all_predictions, references=all_labels)
    
    print(f"Risultati Epoca {epoch + 1}:")
    print(f"  Precision: {results['overall_precision']:.4f}")
    print(f"  Recall:    {results['overall_recall']:.4f}")
    print(f"  F1-Score:  {results['overall_f1']:.4f}")
    print(f"  Accuracy:  {results['overall_accuracy']:.4f}")

print("\nAllenamento completato!")


--- Epoca 1 / 3 ---


Training: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1875/1875 [03:17<00:00,  9.50it/s, loss=0.00241]


Loss media Training: 0.0492


Valutazione: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 486/486 [00:14<00:00, 33.39it/s]


Risultati Epoca 1:
  Precision: 0.8493
  Recall:    0.8757
  F1-Score:  0.8623
  Accuracy:  0.9848

--- Epoca 2 / 3 ---


Training:  45%|█████████████████████████████████████████████████████████████████████████▉                                                                                         | 850/1875 [01:28<01:47,  9.56it/s, loss=0.00127]

KeyboardInterrupt



In [31]:
results = metric.compute(predictions=all_predictions, references=all_labels)
print(f"Risultati Epoca {epoch + 1}:")
print(f"  Precision: {results['overall_precision']:.4f}")
print(f"  Recall:    {results['overall_recall']:.4f}")
print(f"  F1-Score:  {results['overall_f1']:.4f}")
print(f"  Accuracy:  {results['overall_accuracy']:.4f}")

Risultati Epoca 1:
  Precision: 0.8466
  Recall:    0.8602
  F1-Score:  0.8533
  Accuracy:  0.9839


In [ ]:
output_dir = "./model/modello_ner_italiano.pth"
torch.save(model.state_dict(), output_dir)
print(f"Modello salvato in {output_dir}")

In [121]:
from transformers import pipeline

ner_pipline = pipeline(
    "ner", 
    model=model, 
    tokenizer=tokenizer, 
    aggregation_strategy="simple" # Unisce i sub-token (es. "Gari", "##baldi" -> "Garibaldi")
)

test_sentence = "Il Presidente della Repubblica Sergio Mattarella si è recato a Trieste per incontrare i rappresentanti della Commissione Europea."

risultati = ner_pipline(test_sentence)

print("\nEntità trovate:")
for ent in risultati:
    print(f"Testo: {ent['word']:<20} | Etichetta: {ent['entity_group']:<5} | Score: {ent['score']:.4f}")


Entità trovate:
Testo: Sergio Mattarella    | Etichetta: PER   | Score: 0.9968
Testo: Trieste              | Etichetta: LOC   | Score: 0.9969
Testo: Commissione Europea  | Etichetta: ORG   | Score: 0.9985


In [ ]:
#Per ricaricarlo:
# from transformers import AutoModelForTokenClassification, AutoConfig

# # 1. Carichiamo la configurazione originale (serve per sapere quante label avevi)
# config = AutoConfig.from_pretrained("dbmdz/bert-base-italian-xxl-cased", num_labels=len(['B-LOC', 'B-ORG', 'B-PER', 'I-LOC', 'I-ORG', 'I-PER', 'O']))

# # 2. Istanziamo il modello con l'architettura base (pesi iniziali di BERT)
# loaded_model = AutoModelForTokenClassification.from_config(config)

# # 3. Carichiamo i pesi addestrati nel modello
# loaded_model.load_state_dict(torch.load("bert_ner_weights.pt"))

# # 4. Spostiamo sul device corretto
# loaded_model.to(device)
# loaded_model.eval()

# print("Modello ricaricato con successo dai pesi locali.")